In [ ]:
# Problem Statement:
# Imagine Walmart wants to build a Retail Assistant AI that can help store managers make operational decisions.
# For example, a manager at a Bengaluru store asks:
#     “Based on today’s weather and current market demand, which products should I stock more of today?”
# The AI cannot answer reliably using only its existing LLM knowledge because the answer depends on live information such as today's weather and current demand trends.

# So the AI needs to connect to external systems such as:
# a. OpenWeatherMap → current weather
# b. Tavily Search → current market/demand signals
# c. OpenAI LLM → reasoning and final recommendation

# The notebook then asks three important architect-level questions.
# 1. How should the AI connect to external tools — REST API or MCP?
#     At Walmart scale, should we continue connecting agents directly through REST, or standardize tool access through MCP?
# 2. Which framework should orchestrate the AI agent?
#     The notebook implements essentially the same Walmart use case in three different ways: Python-only, LangChain, and LangGraph.
#     Should Walmart build the orchestration using plain Python, LangChain, or LangGraph?
# 3. Should Walmart build each AI component itself or buy/use an existing solution?

In [1]:
# Load the libraries and keys needed for the full notebook.
# This cell also defines the store we will use in every example.
import os
import json
import time
import requests
from typing import TypedDict
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

OPENAI_KEY  = os.getenv('OPENAI_API_KEY')
WEATHER_KEY = os.getenv('OPENWEATHERMAP_API_KEY')
TAVILY_KEY  = os.getenv('TAVILY_API_KEY')

client = OpenAI(api_key=OPENAI_KEY)

# Check whether all required API keys are present before we continue.
missing = [k for k, v in {
    'OPENAI_API_KEY':         OPENAI_KEY,
    'OPENWEATHERMAP_API_KEY': WEATHER_KEY,
    'TAVILY_API_KEY':         TAVILY_KEY,
}.items() if not v]

if missing:
    print(f'WARNING: Missing API keys: {missing}')
    print('Add them to your .env file before running this notebook.')
else:
    print('All required API keys loaded.')

# Fixed store context keeps every comparison in the notebook consistent.
STORE_ID   = 'WMT-2847'
STORE_CITY = 'Bengaluru'
print(f'Store context: {STORE_ID} | {STORE_CITY}, India')

All required API keys loaded.
Store context: WMT-2847 | Bengaluru, India


## The Decision Landscape

Every production AI system at Walmart scale requires three interlocking decisions made before any code is written:

1. **Protocol selection:** How should your AI agent communicate with external systems? REST API or Model Context Protocol (MCP)?
2. **Framework selection:** What orchestration layer do you build on? LangChain, LangGraph, or Python-only?
3. **Build vs Buy:** For each component, is it cheaper to build it or to buy a vendor solution?

Each decision compounds. A wrong protocol choice forces a framework rewrite downstream.

**The running scenario throughout this notebook:**

> *You are the AI engineer for the Walmart India Retail Assistant deployed at 4,700 stores with 50,000+ queries per day. The store manager at WMT-2847 in Bengaluru asks: "Based on today's actual conditions, what should we prioritise stocking today?"*

Every tool call in this notebook returns **live data** from real external APIs. The recommendation changes based on real weather and real market signals retrieved at runtime.

## Core API Functions

These two functions are the live data foundation used across all three sections.
Both make real HTTP calls to external services every time they are invoked.

In [3]:
def fetch_weather(city: str, country_code: str = 'IN') -> dict:
    # Call the live OpenWeatherMap API for current weather.
    url    = 'https://api.openweathermap.org/data/2.5/weather'
    params = {'q': f'{city},{country_code}', 'appid': WEATHER_KEY, 'units': 'metric'}
    resp   = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    d = resp.json()
    print(f"Current weather is {d}")
    return {
        'city':           d['name'],
        'country':        d['sys']['country'],
        'temperature_c':  round(d['main']['temp'], 1),
        'feels_like_c':   round(d['main']['feels_like'], 1),
        'humidity_pct':   d['main']['humidity'],
        'condition':      d['weather'][0]['description'],
        'condition_main': d['weather'][0]['main'],
        'wind_speed_ms':  d['wind']['speed'],
        'pressure_hpa':   d['main']['pressure'],
    }

fetch_weather("Mumbai")

Current weather is {'coord': {'lon': 72.8479, 'lat': 19.0144}, 'weather': [{'id': 500, 'main': 'Rain', 'description': 'light rain', 'icon': '10n'}], 'base': 'stations', 'main': {'temp': 27.94, 'feels_like': 31.95, 'temp_min': 27.94, 'temp_max': 27.94, 'pressure': 1008, 'humidity': 80, 'sea_level': 1008, 'grnd_level': 1008}, 'visibility': 10000, 'wind': {'speed': 1.47, 'deg': 267, 'gust': 1.81}, 'rain': {'1h': 0.36}, 'clouds': {'all': 100}, 'dt': 1790102854, 'sys': {'country': 'IN', 'sunrise': 1790125055, 'sunset': 1790168683}, 'timezone': 19800, 'id': 1275339, 'name': 'Mumbai', 'cod': 200}


{'city': 'Mumbai',
 'country': 'IN',
 'temperature_c': 27.9,
 'feels_like_c': 31.9,
 'humidity_pct': 80,
 'condition': 'light rain',
 'condition_main': 'Rain',
 'wind_speed_ms': 1.47,
 'pressure_hpa': 1008}

In [4]:
def fetch_demand_trends(query: str, max_results: int = 3) -> dict:
    # Call the live Tavily API for current market demand signals.
    resp = requests.post(
        'https://api.tavily.com/search',
        json={
            'api_key':        TAVILY_KEY,
            'query':          query,
            'max_results':    max_results,
            'search_depth':   'basic',
            'include_answer': True,
        },
        timeout=15,
    )
    resp.raise_for_status()
    data = resp.json()
    return {
        'answer':  data.get('answer', ''),
        'results': [
            {'title': r['title'], 'content': r['content'][:350]}
            for r in data.get('results', [])
        ],
    }
fetch_demand_trends("What are the latest updates on US-Israle vs Iran war as of 23rd September 2026?")

{'answer': 'As of September 23, 2026, the US and Israel continue hostilities against Iran, with recent missile exchanges and retaliatory strikes ongoing. Iran has targeted US vessels and oil tankers, while the US destroyed Iranian tankers. No formal ceasefire has been reached.',
 'results': [{'title': '2026 Iran war - Wikipedia',
   'content': "Since 28 February 2026, the United States and Israel have been at war with Iran and its regional allies. Hostilities broke out after US–Israeli airstrikes killed several Iranian officials, including Supreme Leader Ali Khamenei. The strikes were launched amid ongoing US–Iran negotiations regarding Iran's nuclear program. [...] Keir Starmer said the "},
  {'title': 'Conflict With Iran | Global Conflict Tracker',
   'content': 'Overview\n\nIn February 2026, the United States and Israel attacked Iran, killing Supreme Leader Ali Khamenei and targeting nuclear and military infrastructure. A Pakistan-mediated ceasefire and June MOU halted large-scale f